# Masking

Masks let one trained inference network answer several related questions. For example, the same model can infer a full posterior, work with an observation whose second coordinate is missing, return a parameter marginal, or condition on a parameter whose value is already known.

This guide develops those cases with the inverse-kinematics benchmark and then briefly covers padding and attention masks. The examples use a {py:class}`~bayesflow.networks.DiffusionModel` with a {py:class}`~bayesflow.networks.DiffusionTransformer`, but the same semantic-mask interface is also available to {py:class}`~bayesflow.networks.FlowMatching`.

:::{important}
A mask does not give an ordinarily trained network a new skill after the fact. Include the masking patterns during training, then validate every kind of query you plan to use.
:::

## Two kinds of masks

BayesFlow uses two mask families:

- **Semantic masks** change the probabilistic question. They say which target coordinates are inferred, fixed, or marginalized and which condition coordinates were observed.
- **Structural masks** describe tensor structure, such as padding or a custom attention pattern. They route to a summary network or inference subnet but do not, by themselves, redefine the posterior.

Most of this guide concerns semantic masks.

## Semantic masks at a glance

All semantic masks follow a **keep-mask convention**: `1` means that a coordinate is active or present, while `0` removes it from its usual role.

| Argument | Applies to | `1` means | `0` means |
|---|---|---|---|
| `observed_condition_mask` | conditions $\mathbf{x}$ | observed and usable | missing; ignore this condition |
| `infer_target_mask` | targets $\boldsymbol\theta$ | present in the requested target | absent; marginalize this target |
| `fixed_target_mask` | targets $\boldsymbol\theta$ | infer and update | hold fixed at `fixed_target_value` |

`fixed_target_value` is not a mask. It is the complete target-shaped value array used wherever `fixed_target_mask` is `0`.

For ordinary vector data, use shape `(num_datasets, num_dimensions)`. There is no posterior-sample axis: BayesFlow broadcasts each row over `num_samples`. A shared one-dimensional mask is also accepted in many low-level calls, but keeping the data-batch axis makes examples and batched inference unambiguous.

When no semantic mask is supplied, all conditions are observed and all targets are inferred.

### What changes statistically?

Suppose $A$ and $B$ partition the targets and $O$ and $M$ partition the observation:

- **Missing observation:** `observed_condition_mask` requests $p(\boldsymbol\theta\mid\mathbf{x}_O)$. The unavailable $\mathbf{x}_M$ is not treated as an observation equal to zero; it is excluded as evidence.
- **Target marginal:** `infer_target_mask` requests $p(\boldsymbol\theta_A\mid\mathbf{x})=\int p(\boldsymbol\theta_A,\boldsymbol\theta_B\mid\mathbf{x})\,d\boldsymbol\theta_B$.
- **Arbitrary conditional:** `fixed_target_mask` together with `fixed_target_value` requests $p(\boldsymbol\theta_A\mid\boldsymbol\theta_B=\mathbf{b},\mathbf{x})$.

These cases are easy to mix up. A missing observation removes information on the **conditioning side**. A target marginal removes a variable from the **quantity being inferred**. A fixed target instead adds its known value to the conditioning information.

## Why use a DiffusionTransformer?

The `DiffusionTransformer` turns every target and condition dimension into its own token. Each token receives an identity embedding and one of three learned state embeddings:

1. **latent** — a target currently being inferred,
2. **observed** — a fixed target or an available condition,
3. **missing** — a marginalized target or unavailable condition.

The transformer derives a directed attention mask from those states. Inferred target tokens can use all present information; fixed target tokens act as clean context and receive no residual-block updates; missing tokens cannot become evidence. The outer `DiffusionModel` or `FlowMatching` supplies clean fixed values and projects integration states and updates. Diffusion time modulates every block through adaptive layer normalization. If an explicit attention mask is also supplied, it is intersected with the semantic mask, so custom attention cannot make a missing token visible again.

Despite its name, this is a **subnet architecture**, not a separate inference algorithm. Use `subnet="diffusion_transformer"` with `DiffusionModel` or `FlowMatching` when you need per-dimension semantic masking. A plain MLP receives a flat vector and cannot enforce the same token-level separation.

## Running example: inverse kinematics

The benchmark describes a planar robot arm with target vector

$$\boldsymbol\theta=(h,\alpha_1,\alpha_2,\alpha_3)$$

and observed endpoint $\mathbf{x}=(x_1,x_2)$. It is a useful example because we can ask intuitive variants of the same inverse problem: infer all four arm parameters, ignore a broken position sensor, marginalize the height, or fix the height to a measured value.

The scenarios mirror the [inverse-kinematics masking example](https://github.com/bayesflow-org/diffusion-experiments/blob/main/intro_example/plot_transformer_conditionals.py) from the diffusion experiments.

In [ ]:
import bayesflow as bf
import numpy as np

rng = np.random.default_rng(2026)
simulator = bf.simulators.InverseKinematics(rng=rng)

adapter = (
    bf.adapters.Adapter()
    .to_array()
    .rename("parameters", "inference_variables")
    .rename("observables", "inference_conditions")
)

# One observed arm endpoint; the leading axis indexes datasets.
observation = {"observables": np.array([[0.0, 1.5]], dtype=np.float32)}

## Train one model for all cases

Three probabilities augment each training batch with random masking patterns:

- `fixed_target_prob` moves some target coordinates from the inferred set to the clean conditioning set. This teaches arbitrary target conditionals. The automatic generator keeps at least one eligible target inferred in each row.
- `missing_target_prob` hides some coordinates that would otherwise be fixed. This teaches target marginals. For automatic mask generation, set `fixed_target_prob > 0` when using it; an explicit training-time `fixed_target_mask` can provide the candidates instead.
- `missing_conditions_prob` hides condition coordinates. This teaches inference from incomplete observations.

The values below are a reasonable illustration, not universal tuning constants. Increasing `fixed_target_prob` reduces ordinary full-posterior rows, and increasing `missing_conditions_prob` reduces complete-observation rows. By contrast, `missing_target_prob` changes the mix of fixed versus marginalized coordinates only within target queries that already contain fixed coordinates.

:::{tip}
To use flow matching, replace `DiffusionModel` with `FlowMatching` in the next cell. The three training probabilities and every inference-time mask remain the same.
:::

In [ ]:
inference_network = bf.networks.DiffusionModel(
    subnet="diffusion_transformer",
    fixed_target_prob=0.3,
    missing_target_prob=0.3,
    missing_conditions_prob=0.3,
    seed=2026,  # local generator owned by this network
)

workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    inference_network=inference_network,
    standardize="inference_conditions",
)

Train exactly as for an unmasked amortized posterior. Random semantic masks are sampled only during training; the explicit masks in later sections select a query at inference time.

The following is the only expensive cell. The simulation count and epochs are deliberately visible so they can be adjusted for an experiment.

In [ ]:
training_data = simulator.sample(10_000)
history = workflow.fit_offline(training_data, epochs=200, batch_size=128)

## Reference: the full posterior

With no masks, both endpoint coordinates are observed and all four arm parameters are inferred. The returned array has shape `(1, 1000, 4)`: one dataset, 1000 posterior draws, and four targets.

In [ ]:
posterior_full = workflow.sample(
    conditions=observation,
    num_samples=1_000,
    seed=11,
)["parameters"]

## Missing observations

Suppose $x_2$ was not recorded. Keep $x_1$ with a `1` and mark $x_2$ missing with a `0`. The value stored in the masked position is ignored by the transformer, but it should still be a finite placeholder rather than `NaN` so preprocessing remains well behaved.

This query approximates $p(h,\alpha_1,\alpha_2,\alpha_3\mid x_1)$, which will usually be broader than the full posterior because it contains less evidence.

In [ ]:
condition_observed = np.array([[1.0, 0.0]], dtype=np.float32)

posterior_missing_x2 = workflow.sample(
    conditions=observation,
    num_samples=1_000,
    observed_condition_mask=condition_observed,
    seed=12,
)["parameters"]

## Target marginals

Now keep the complete endpoint but request only the three angle coordinates. Setting the height entry of `infer_target_mask` to `0` tells the model to integrate out $h$ rather than condition on a particular height.

The returned tensor keeps the original four-coordinate layout. The value in a marginalized coordinate is only an implementation placeholder; it is **not** a draw from that coordinate's prior or marginal posterior. Discard masked coordinates explicitly, as the final line does below.

In [ ]:
target_present = np.array([[0.0, 1.0, 1.0, 1.0]], dtype=np.float32)

posterior_without_height = workflow.sample(
    conditions=observation,
    num_samples=1_000,
    infer_target_mask=target_present,
    seed=13,
)["parameters"]

angle_samples = posterior_without_height[..., 1:]

## Fixed targets and arbitrary conditionals

A fixed target is different from a marginalized target. Here we know that $h=0$ and want $p(\alpha_1,\alpha_2,\alpha_3\mid h=0,\mathbf{x})$.

`fixed_target_mask` uses `0` for the fixed coordinate and `1` for coordinates that remain free. `fixed_target_value` must have the complete target shape; entries selected by `1` are ignored. At the workflow boundary, supply `fixed_target_value` in the adapted target coordinates, before workflow standardization. Here the adapter only renames, so those are the original parameter units; BayesFlow applies configured workflow standardization internally. During diffusion or flow-matching integration, BayesFlow zeros learned updates for fixed coordinates; stochastic solvers also project their states and noise updates so those coordinates cannot drift.

In [ ]:
target_free = np.array([[0.0, 1.0, 1.0, 1.0]], dtype=np.float32)
target_value = np.array([[0.0, 0.0, 0.0, 0.0]], dtype=np.float32)

posterior_given_zero_height = workflow.sample(
    conditions=observation,
    num_samples=1_000,
    fixed_target_mask=target_free,
    fixed_target_value=target_value,
    seed=14,
)["parameters"]

## An all-in-one query

The masks can be combined when they refer to distinct roles. In the next query:

- $h$ is present but fixed to zero,
- $\alpha_1$ and $\alpha_2$ are inferred,
- $\alpha_3$ is marginalized,
- $x_1$ is observed and $x_2$ is missing.

In probability notation, the useful output approximates $p(\alpha_1,\alpha_2\mid h=0,x_1)$ with $\alpha_3$ integrated out.

In [ ]:
combined = workflow.sample(
    conditions=observation,
    num_samples=1_000,
    observed_condition_mask=np.array([[1.0, 0.0]], dtype=np.float32),
    infer_target_mask=np.array([[1.0, 1.0, 1.0, 0.0]], dtype=np.float32),
    fixed_target_mask=np.array([[0.0, 1.0, 1.0, 1.0]], dtype=np.float32),
    fixed_target_value=np.zeros((1, 4), dtype=np.float32),
    seed=15,
)["parameters"]

alpha_1_and_2 = combined[..., 1:3]  # discard the marginalized alpha_3 slot

## The same masks apply to densities

For inference networks that implement `log_prob`, semantic masks have the same meaning during density evaluation. For example, the call below evaluates the learned three-angle marginal density at eight simulated points, using one Hutchinson probe for the divergence estimate. Fixed target coordinates are similarly excluded from the reported conditional density.

In [ ]:
evaluation_data = simulator.sample(8)
angle_log_prob = workflow.log_prob(
    evaluation_data,
    infer_target_mask=target_present,
    hutchinson_steps=1,
    seed=16,
)

## Structural masks: padding and attention

Structural masks solve a different problem. They describe which positions in a padded set or sequence are valid, or which tokens may attend to one another.

| Workflow/adapted-data key | Routed to | Typical use |
|---|---|---|
| `summary_mask` | summary network's `mask` | mark valid entries in padded summary data |
| `summary_attention_mask` | summary network's `attention_mask` | control attention among summary tokens |
| `inference_mask` | inference subnet's `mask` | mark valid entries in structured inference data |
| `inference_attention_mask` | inference subnet's `attention_mask` | control inference-subnet attention |

Configure the adapter to produce or preserve both `summary_variables` and `summary_mask`. A sampling call can then look like:

```python
conditions = {
    "summary_variables": padded_trials,
    "summary_mask": valid_trials,
}
samples = padded_workflow.sample(conditions=conditions, num_samples=1_000)
```

A receiving subnet must explicitly support `mask` or `attention_mask`. BayesFlow raises when a structural mask would otherwise be silently ignored; coupling layers, for example, do not pass structural masks to their internal subnets.

The `DiffusionTransformer` also accepts a direct attention mask with shape `(batch, num_targets, num_targets + num_conditions)`. Semantic visibility is always enforced in addition to that custom pattern.

## Practical checklist

- Train on every kind of masking pattern you intend to query and validate each one separately.
- Keep a leading dataset axis on masks and align dimensions with the **adapted** target or condition order.
- Use finite placeholders for missing observations; the mask, not a sentinel value, communicates missingness.
- Discard output coordinates where `infer_target_mask == 0`.
- Supply a full target-shaped `fixed_target_value` and remember that `fixed_target_mask == 0` means fixed.
- Explicit call-time masks override masks found in adapted data. This is useful for one-off queries, but avoid defining the same mask in two places without a reason.
- Use `summary_*` and `inference_*` names at workflow boundaries; subnets receive the canonical `mask` and `attention_mask` names.

## Summary

The shortest reliable mental model is:

- `observed_condition_mask` says what evidence exists,
- `infer_target_mask` says which random variables appear in the requested output distribution,
- `fixed_target_mask` and `fixed_target_value` say which present targets are known,
- structural masks describe valid tensor positions and attention edges.

Keeping those roles separate makes even a combined masked query readable.